# 03 — Compute EGFR structural features

This notebook converts the generated EGFR mutant structures into numerical features for machine learning.

It will:

1. upload `EGFR_mutant_structure_bundle.zip`,
2. extract the reference and mutant PDB files,
3. read the mutation-generation report,
4. identify the bound inhibitor in each reference structure,
5. calculate chemistry, distance, packing, and contact-change features,
6. merge those features with the experimental AUC-ratio outcome, and
7. download `egfr_structural_features.csv`.

## Important interpretation

These structures have not yet undergone ligand-aware energy minimization. Therefore, the features generated here are best viewed as **fast structural descriptors of the modeled substitution**, not fully relaxed binding free-energy predictions.

## Step 1 — Upload the ZIP file

Upload:

`EGFR_mutant_structure_bundle.zip`


In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded))

## Step 2 — Install required packages

In [ ]:
!pip -q install pandas numpy biopython scipy

## Step 3 — Extract the bundle

In [ ]:
from pathlib import Path
import zipfile
import shutil
import os

zip_name = "EGFR_mutant_structure_bundle.zip"
extract_dir = Path("EGFR_bundle")

if extract_dir.exists():
    shutil.rmtree(extract_dir)

with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall(extract_dir)

print("Extracted files:")
for path in sorted(extract_dir.rglob("*")):
    if path.is_file():
        print(path)

## Step 4 — Load and inspect the generation report

In [ ]:
import pandas as pd

report_path = extract_dir / "mutant_generation_report.csv"
report = pd.read_csv(report_path)

generated = report[
    (report["generation_status"] == "generated") &
    (report["mutation_verified"] == True)
].copy()

print("Rows in full report:", len(report))
print("Verified generated pairs:", len(generated))
print("Drugs:", sorted(generated["drug"].unique()))
display(generated.head(10))

## Step 5 — Define amino-acid properties

These descriptors capture the intrinsic chemical effect of replacing one amino acid with another.

In [ ]:
AA_VOLUME = {
    "A": 88.6, "R": 173.4, "N": 114.1, "D": 111.1, "C": 108.5,
    "Q": 143.8, "E": 138.4, "G": 60.1, "H": 153.2, "I": 166.7,
    "L": 166.7, "K": 168.6, "M": 162.9, "F": 189.9, "P": 112.7,
    "S": 89.0, "T": 116.1, "W": 227.8, "Y": 193.6, "V": 140.0,
}

AA_HYDROPATHY = {
    "A": 1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C": 2.5,
    "Q": -3.5, "E": -3.5, "G": -0.4, "H": -3.2, "I": 4.5,
    "L": 3.8, "K": -3.9, "M": 1.9, "F": 2.8, "P": -1.6,
    "S": -0.8, "T": -0.7, "W": -0.9, "Y": -1.3, "V": 4.2,
}

AA_CHARGE = {
    "A": 0, "R": 1, "N": 0, "D": -1, "C": 0,
    "Q": 0, "E": -1, "G": 0, "H": 0.1, "I": 0,
    "L": 0, "K": 1, "M": 0, "F": 0, "P": 0,
    "S": 0, "T": 0, "W": 0, "Y": 0, "V": 0,
}

AA_AROMATIC = {aa: int(aa in {"F", "W", "Y", "H"}) for aa in AA_VOLUME}
AA_POLAR = {aa: int(aa in {"R", "N", "D", "Q", "E", "H", "K", "S", "T", "Y", "C"}) for aa in AA_VOLUME}

## Step 6 — Structural helper functions

The bound inhibitor is selected as the largest non-water hetero-residue in the PDB. The code then measures:

- mutation-to-drug distance,
- mutation–drug heavy-atom contacts,
- local residue packing,
- contacts gained or lost after mutation, and
- local backbone RMSD.


In [ ]:
import re
import numpy as np
from Bio.PDB import PDBParser, Superimposer
from scipy.spatial.distance import cdist

parser = PDBParser(QUIET=True)

WATER_NAMES = {"HOH", "WAT", "DOD"}
COMMON_IONS = {
    "NA", "K", "CL", "MG", "CA", "ZN", "MN", "FE", "CU", "CO",
    "SO4", "PO4", "GOL", "EDO"
}

def parse_mutation(mutation):
    match = re.fullmatch(r"([A-Z])(\d+)([A-Z])", str(mutation).strip().upper())
    if not match:
        raise ValueError(f"Unsupported mutation: {mutation}")
    wt, position, mut = match.groups()
    return wt, int(position), mut

def load_first_model(pdb_path):
    structure = parser.get_structure("structure", str(pdb_path))
    return next(structure.get_models())

def heavy_atoms(residue):
    return [
        atom for atom in residue.get_atoms()
        if atom.element != "H"
    ]

def protein_residues(model):
    residues = []
    for chain in model:
        for residue in chain:
            if residue.id[0] == " ":
                residues.append((chain.id, residue))
    return residues

def find_residue(model, chain_id, position):
    for chain in model:
        if chain.id != chain_id:
            continue
        for residue in chain:
            if residue.id[0] == " " and residue.id[1] == int(position):
                return residue
    raise KeyError(f"Residue {chain_id}:{position} not found")

def identify_ligand(model):
    candidates = []
    for chain in model:
        for residue in chain:
            hetflag = residue.id[0]
            name = residue.get_resname().strip().upper()
            atoms = heavy_atoms(residue)

            if hetflag == " ":
                continue
            if name in WATER_NAMES or name in COMMON_IONS:
                continue
            if len(atoms) < 5:
                continue

            candidates.append((len(atoms), chain.id, residue))

    if not candidates:
        raise ValueError("No suitable inhibitor-like hetero-residue found")

    candidates.sort(key=lambda x: x[0], reverse=True)
    _, chain_id, residue = candidates[0]
    return chain_id, residue

def atom_coordinates(atoms):
    return np.array([atom.coord for atom in atoms], dtype=float)

def minimum_distance(atoms_a, atoms_b):
    if not atoms_a or not atoms_b:
        return np.nan
    return float(cdist(atom_coordinates(atoms_a), atom_coordinates(atoms_b)).min())

def contact_count(atoms_a, atoms_b, cutoff=4.5):
    if not atoms_a or not atoms_b:
        return 0
    distances = cdist(atom_coordinates(atoms_a), atom_coordinates(atoms_b))
    return int((distances <= cutoff).sum())

def contacting_residue_ids(model, target_residue, cutoff=4.5):
    target_atoms = heavy_atoms(target_residue)
    contacts = set()

    for chain_id, residue in protein_residues(model):
        if residue is target_residue:
            continue
        if minimum_distance(target_atoms, heavy_atoms(residue)) <= cutoff:
            contacts.add((chain_id, residue.id[1], residue.get_resname()))

    return contacts

def local_neighbor_count(model, target_residue, cutoff=8.0):
    if "CA" not in target_residue:
        return np.nan

    center = target_residue["CA"].coord
    count = 0

    for _, residue in protein_residues(model):
        if residue is target_residue or "CA" not in residue:
            continue
        if np.linalg.norm(residue["CA"].coord - center) <= cutoff:
            count += 1

    return count

def local_backbone_rmsd(wt_model, mut_model, chain_id, position, radius=8.0):
    wt_target = find_residue(wt_model, chain_id, position)
    if "CA" not in wt_target:
        return np.nan

    center = wt_target["CA"].coord
    wt_atoms = []
    mut_atoms = []

    for chain in wt_model:
        if chain.id != chain_id:
            continue
        for residue in chain:
            if residue.id[0] != " " or "CA" not in residue:
                continue
            if np.linalg.norm(residue["CA"].coord - center) > radius:
                continue

            try:
                mut_residue = find_residue(mut_model, chain_id, residue.id[1])
            except KeyError:
                continue

            for atom_name in ("N", "CA", "C"):
                if atom_name in residue and atom_name in mut_residue:
                    wt_atoms.append(residue[atom_name])
                    mut_atoms.append(mut_residue[atom_name])

    if len(wt_atoms) < 3:
        return np.nan

    superimposer = Superimposer()
    superimposer.set_atoms(wt_atoms, mut_atoms)
    return float(superimposer.rms)

## Step 7 — Calculate features for all verified mutation–drug pairs

In [ ]:
reference_dir = extract_dir / "reference_structures"
mutant_dir = extract_dir / "mutant_structures"

feature_rows = []

for row in generated.itertuples(index=False):
    wt_aa, position, mut_aa = parse_mutation(row.mutation)

    reference_pdb = reference_dir / f"{row.pdb_id}.pdb"
    mutant_pdb = mutant_dir / Path(row.output_pdb).name

    wt_model = load_first_model(reference_pdb)
    mut_model = load_first_model(mutant_pdb)

    wt_residue = find_residue(wt_model, row.chain, position)
    mut_residue = find_residue(mut_model, row.chain, position)

    ligand_chain, wt_ligand = identify_ligand(wt_model)
    _, mut_ligand = identify_ligand(mut_model)

    wt_sidechain = [
        atom for atom in heavy_atoms(wt_residue)
        if atom.name not in {"N", "CA", "C", "O"}
    ]
    mut_sidechain = [
        atom for atom in heavy_atoms(mut_residue)
        if atom.name not in {"N", "CA", "C", "O"}
    ]

    wt_ligand_atoms = heavy_atoms(wt_ligand)
    mut_ligand_atoms = heavy_atoms(mut_ligand)

    wt_contacts = contacting_residue_ids(wt_model, wt_residue)
    mut_contacts = contacting_residue_ids(mut_model, mut_residue)

    feature_rows.append({
        "drug": row.drug,
        "pdb_id": row.pdb_id,
        "mutation": row.mutation,
        "chain": row.chain,
        "position": position,
        "wt_aa": wt_aa,
        "mut_aa": mut_aa,

        # Experimental targets
        "auc_ratio_vs_wt": row.auc_ratio_vs_wt,
        "resistant": row.resistant,

        # Mutation chemistry
        "delta_sidechain_volume_A3":
            AA_VOLUME[mut_aa] - AA_VOLUME[wt_aa],
        "delta_hydropathy":
            AA_HYDROPATHY[mut_aa] - AA_HYDROPATHY[wt_aa],
        "delta_charge":
            AA_CHARGE[mut_aa] - AA_CHARGE[wt_aa],
        "delta_aromatic":
            AA_AROMATIC[mut_aa] - AA_AROMATIC[wt_aa],
        "delta_polar":
            AA_POLAR[mut_aa] - AA_POLAR[wt_aa],

        # WT geometry
        "wt_mutation_to_drug_min_A":
            minimum_distance(wt_sidechain, wt_ligand_atoms),
        "wt_mutation_drug_atom_contacts_4p5A":
            contact_count(wt_sidechain, wt_ligand_atoms, 4.5),
        "wt_local_neighbors_8A":
            local_neighbor_count(wt_model, wt_residue, 8.0),
        "wt_local_residue_contacts_4p5A":
            len(wt_contacts),

        # Mutant geometry
        "mut_mutation_to_drug_min_A":
            minimum_distance(mut_sidechain, mut_ligand_atoms),
        "mut_mutation_drug_atom_contacts_4p5A":
            contact_count(mut_sidechain, mut_ligand_atoms, 4.5),
        "mut_local_neighbors_8A":
            local_neighbor_count(mut_model, mut_residue, 8.0),
        "mut_local_residue_contacts_4p5A":
            len(mut_contacts),

        # Structural changes
        "delta_mutation_to_drug_min_A":
            minimum_distance(mut_sidechain, mut_ligand_atoms)
            - minimum_distance(wt_sidechain, wt_ligand_atoms),
        "delta_mutation_drug_atom_contacts_4p5A":
            contact_count(mut_sidechain, mut_ligand_atoms, 4.5)
            - contact_count(wt_sidechain, wt_ligand_atoms, 4.5),
        "delta_local_neighbors_8A":
            local_neighbor_count(mut_model, mut_residue, 8.0)
            - local_neighbor_count(wt_model, wt_residue, 8.0),
        "contacts_lost_4p5A":
            len(wt_contacts - mut_contacts),
        "contacts_gained_4p5A":
            len(mut_contacts - wt_contacts),
        "local_backbone_rmsd_A":
            local_backbone_rmsd(
                wt_model, mut_model, row.chain, position, radius=8.0
            ),

        # Ligand identity check
        "ligand_resname": wt_ligand.get_resname(),
        "ligand_chain": ligand_chain,
        "ligand_heavy_atoms": len(wt_ligand_atoms),
    })

features = pd.DataFrame(feature_rows)

print("Feature rows:", len(features))
print("Columns:", len(features.columns))
display(features.head(10))

## Step 8 — Sanity checks

The number of feature rows should equal the number of verified generated mutation–drug pairs.

In [ ]:
assert len(features) == len(generated), (
    f"Expected {len(generated)} rows but calculated {len(features)}"
)

print("Rows by drug:")
display(features.groupby("drug").size().rename("rows"))

print("\nLigand selected for each structure:")
display(
    features[
        ["drug", "pdb_id", "ligand_resname", "ligand_chain",
         "ligand_heavy_atoms"]
    ].drop_duplicates()
)

print("\nMissing values by column:")
missing = features.isna().sum()
display(missing[missing > 0].sort_values(ascending=False))

## Step 9 — Save and download the machine-learning feature table

This output contains one row per supported mutation–drug pair, the experimental outcome, and the computed structural features.

In [ ]:
output_file = "egfr_structural_features.csv"
features.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Shape:", features.shape)

files.download(output_file)

## Next notebook

The next notebook will train a regression model to predict `auc_ratio_vs_wt`, evaluate it using mutation-grouped cross-validation, compare it with a classification model, and rank which structural features are most predictive of resistance.